# Text-to-SVG Inference v2 — Advanced Submission

**Improvements over v1:**
- Correct viewBox handling: preserves model's natural coordinate space, only sets width=height=256
- Retry logic: invalid SVG → retry with temperature sampling
- SVG repair: closes unclosed tags on truncated outputs
- Progress checkpointing: saves partial results every 50 rows
- Versioned package pins matching training environment

In [ ]:
%pip install -q unsloth==2026.3.8 transformers==5.3.0 peft==0.18.1 \
    bitsandbytes==0.49.2 accelerate==1.13.0 trl==0.24.0 \
    cairosvg lxml

In [ ]:
import os, re, time, random, json
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────────
BASE_MODEL_PATH  = "/kaggle/input/svg-base-model"
LORA_PATH        = "/kaggle/input/svg-lora-model/lora"
TEST_CSV         = "/kaggle/input/svg-competition/test.csv"
SUBMISSION_PATH  = "/kaggle/working/submission.csv"
CHECKPOINT_PATH  = "/kaggle/working/submission_checkpoint.csv"  # partial saves

# Generation config
# p95 SVG ≈ 1760 tokens, max ≈ 4553 tokens at ~3.5 chars/token
MAX_NEW_TOKENS   = 3000
MAX_RETRY_TOKENS = 2000   # shorter for retry pass
SAVE_EVERY       = 50     # checkpoint every N rows

print("Paths configured.")

In [ ]:
# ── SVG constants & helpers ─────────────────────────────────────────────────────
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clippath", "mask", "lineargradient",
    "radialgradient", "stop", "text", "tspan", "title",
    "desc", "style", "pattern", "marker", "filter",
}
DISALLOWED_TAGS = {
    "script", "animate", "animatetransform", "animatemotion",
    "animatecolor", "set", "foreignobject",
}
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", re.IGNORECASE)
SVG_OPEN  = re.compile(r"<svg[\s\S]*", re.IGNORECASE)

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
    '<rect x="0" y="0" width="256" height="256" fill="white"/>'
    '<circle cx="128" cy="128" r="80" fill="#4A90D9"/>'
    '</svg>'
)


def fix_svg(svg_text: str) -> str:
    """
    Normalize SVG:
    - Set width=256, height=256 (canvas size)
    - PRESERVE the model's natural viewBox (training data is 200x200 coordinate space)
    - Add xmlns if missing
    - If viewBox missing, default to 0 0 200 200 (training data median)
    """
    if len(svg_text) > 16000:
        svg_text = svg_text[:16000]
    if 'xmlns=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    # Replace existing width/height values with 256
    svg_text = re.sub(r'\bwidth=["\'][^"\']*["\']', 'width="256"', svg_text, count=1)
    svg_text = re.sub(r'\bheight=["\'][^"\']*["\']', 'height="256"', svg_text, count=1)
    if 'width=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg width="256"', 1)
    if 'height=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg height="256"', 1)
    # Preserve viewBox — only add if missing
    if 'viewBox=' not in svg_text and 'viewbox=' not in svg_text.lower():
        svg_text = svg_text.replace("<svg", '<svg viewBox="0 0 200 200"', 1)
    return svg_text


def repair_svg(svg_text: str) -> str:
    """
    Try to repair a truncated SVG by closing any open tags.
    Works on outputs cut off by max_new_tokens.
    """
    # Find opening tag stack
    open_tags = re.findall(r'<([a-zA-Z][a-zA-Z0-9]*)[\s>]', svg_text)
    self_closing = {'path','rect','circle','ellipse','line','polyline',
                    'polygon','stop','use','desc','title'}
    # Find closed tags
    close_tags = re.findall(r'</([a-zA-Z][a-zA-Z0-9]*)>', svg_text)
    closed = set(close_tags)

    # If truncated mid-tag, remove the dangling partial tag
    svg_text = re.sub(r'<[^>]*$', '', svg_text)

    # Close unclosed container tags in reverse order
    stack = []
    for tag in open_tags:
        tl = tag.lower()
        if tl in self_closing:
            continue
        stack.append(tl)

    for tag in close_tags:
        tl = tag.lower()
        if tl in stack:
            stack.remove(tl)

    # Append closing tags for anything still open (reverse order)
    for tag in reversed(stack):
        svg_text += f'</{tag}>'

    # Ensure </svg> at end
    if not svg_text.rstrip().endswith('</svg>'):
        svg_text += '</svg>'

    return svg_text


def is_valid_svg(svg_text: str) -> bool:
    if not svg_text or len(svg_text) > 16000:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if not root.tag.lower().endswith("svg"):
        return False
    path_count = 0
    for elem in root.iter():
        local = (elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag).lower()
        if local in DISALLOWED_TAGS:
            return False
        if local not in ALLOWED_TAGS:
            return False
        for attr in elem.attrib:
            if attr.lower().startswith("on"):
                return False
        for attr_name in ("href", "{http://www.w3.org/1999/xlink}href"):
            val = elem.attrib.get(attr_name, "")
            if val.startswith("http") or val.startswith("//"):
                return False
        if local == "path":
            path_count += 1
    return path_count <= 256


def extract_and_validate_svg(raw_text: str) -> tuple[str, bool]:
    """Extract → fix → validate. Try repair if invalid."""
    # Try full match first
    m = SVG_REGEX.search(raw_text)
    if m:
        svg = fix_svg(m.group(0).strip())
        if is_valid_svg(svg):
            return svg, True

    # Try repair on truncated output (opening <svg> found but no closing tag)
    m2 = SVG_OPEN.search(raw_text)
    if m2:
        svg = fix_svg(repair_svg(m2.group(0).strip()))
        if is_valid_svg(svg):
            return svg, True

    return FALLBACK_SVG, False


print("SVG helpers loaded.")

In [ ]:
# ── Load model ──────────────────────────────────────────────────────────────────
import unsloth  # noqa: F401 — must be first
from unsloth import FastLanguageModel
from peft import PeftModel

print(f"Loading base model from: {BASE_MODEL_PATH}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_PATH,
    max_seq_length=8192,
    dtype=None,
    load_in_4bit=True,
)

print(f"Applying LoRA adapter from: {LORA_PATH}")
model = PeftModel.from_pretrained(model, LORA_PATH)

FastLanguageModel.for_inference(model)
print("Model ready.")

In [ ]:
# ── Generation ──────────────────────────────────────────────────────────────────
# System prompt matches training exactly
SYSTEM_PROMPT = (
    "You are an expert SVG code generator. "
    "Given a text description, output only valid SVG markup. "
    "Rules:\n"
    "- Root element must be <svg> with xmlns=\"http://www.w3.org/2000/svg\", "
    "width=\"256\", height=\"256\", and a viewBox matching the coordinate space used\n"
    "- Use only these elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, "
    "defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, "
    "title, desc, style, pattern, marker, filter\n"
    "- No scripts, no animation, no event handlers, no external references\n"
    "- Maximum 16000 characters, maximum 256 path elements\n"
    "- Match the visual complexity of the description — do not over-simplify or over-elaborate\n"
    "- Output only the SVG code, nothing else"
)


def build_prompt(user_text: str) -> str:
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_text}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


def _generate_raw(prompt_text: str, max_tokens: int, do_sample: bool = False,
                  temperature: float = 0.7) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    kwargs = dict(
        max_new_tokens=max_tokens,
        do_sample=do_sample,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        kwargs["temperature"] = temperature
        kwargs["top_p"] = 0.9
    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def generate_svg(prompt: str) -> tuple[str, bool, int]:
    """
    Generate SVG with retry logic.
    Pass 1: greedy decoding (deterministic, best SSIM)
    Pass 2: if invalid, retry with light temperature sampling
    Pass 3: if still invalid, use repair heuristic on greedy output
    Returns (svg, is_valid, attempts_used)
    """
    chat = build_prompt(prompt)

    # Pass 1: greedy
    raw1 = _generate_raw(chat, MAX_NEW_TOKENS, do_sample=False)
    svg, valid = extract_and_validate_svg(raw1)
    if valid:
        return svg, True, 1

    # Pass 2: sample with low temperature
    raw2 = _generate_raw(chat, MAX_RETRY_TOKENS, do_sample=True, temperature=0.3)
    svg2, valid2 = extract_and_validate_svg(raw2)
    if valid2:
        return svg2, True, 2

    # Pass 3: return best repaired from pass 1 (or fallback)
    return svg, False, 3


# Smoke test
test_svg, valid, attempts = generate_svg("a simple blue circle on white background")
print(f"Smoke test — valid: {valid} | attempts: {attempts}")
print(test_svg[:300])

In [ ]:
# ── Inference with checkpointing ────────────────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
print(f"Test samples: {len(test_df)}")

# Resume from checkpoint if it exists
rows = []
done_ids = set()
if Path(CHECKPOINT_PATH).exists():
    ckpt = pd.read_csv(CHECKPOINT_PATH)
    rows = ckpt.to_dict('records')
    done_ids = set(ckpt['id'])
    print(f"Resuming from checkpoint — {len(done_ids)} already done")

fallback_count = 0
retry_count    = 0
t0 = time.time()

for i, row in test_df.iterrows():
    if row['id'] in done_ids:
        continue

    svg, valid, attempts = generate_svg(str(row['prompt']))
    if not valid:
        fallback_count += 1
    if attempts > 1:
        retry_count += 1
    rows.append({'id': row['id'], 'svg': svg})

    # Checkpoint save
    if len(rows) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(CHECKPOINT_PATH, index=False)

    if (i + 1) % 25 == 0:
        elapsed = (time.time() - t0) / 60
        done = len(rows)
        eta = (elapsed / done) * (len(test_df) - done) if done else 0
        print(f"  [{done}/{len(test_df)}] {elapsed:.1f}m elapsed | ETA {eta:.1f}m "
              f"| fallbacks: {fallback_count} | retries: {retry_count}")

elapsed_total = (time.time() - t0) / 60
print(f"\nDone. {elapsed_total:.1f} min | Fallbacks: {fallback_count} | Retries: {retry_count}")

In [ ]:
# ── Save & validate submission ───────────────────────────────────────────────────
sub_df = pd.DataFrame(rows)

# Final validation pass
invalid_mask = sub_df['svg'].apply(lambda s: not is_valid_svg(s))
print(f"Invalid SVGs before final fix: {invalid_mask.sum()}")
sub_df.loc[invalid_mask, 'svg'] = FALLBACK_SVG

sub_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved: {SUBMISSION_PATH}")
print(f"Shape: {sub_df.shape}")
print(f"Final invalid count (must be 0): {sub_df['svg'].apply(lambda s: not is_valid_svg(s)).sum()}")
print(sub_df.head(3))